# 01 · Basics: SELECT, WHERE, ORDER BY, LIMIT

**Dataset:** 1,000 synthetic healthcare claims (January–June 2026). All data is fake.

**Tables:** `claims`, `patients`, `providers`, `procedures`.

> 🇪🇸 **Para Josué:** este es el primer notebook de tu portafolio. Los enunciados están en inglés, como en una empresa de EE. UU.; debajo de cada uno va la traducción en un recuadro como este (lo quitamos antes de publicar).
> Hay **4 tablas** (las ves abajo). La primera vez, corre todo con **Run All**. **Guarda con Ctrl + S después de cada ejercicio.**

In [1]:
# Load the CSV files as tables.
import duckdb

duckdb.sql("CREATE OR REPLACE TABLE claims     AS SELECT * FROM '../data/claims.csv'")
duckdb.sql("CREATE OR REPLACE TABLE patients   AS SELECT * FROM '../data/patients.csv'")
duckdb.sql("CREATE OR REPLACE TABLE providers  AS SELECT * FROM '../data/providers.csv'")
duckdb.sql("CREATE OR REPLACE TABLE procedures AS SELECT * FROM '../data/procedures.csv'")
print("Tables loaded: claims, patients, providers, procedures")

Tables loaded: claims, patients, providers, procedures


## The 4 tables

> 🇪🇸 `claims` tiene 1,000 filas y `patients` tiene 200: son demasiadas para verlas todas, así que muestro solo las **primeras 10** con `LIMIT 10` (lo explico más abajo).

### `claims` (1,000 rows)

In [2]:
duckdb.sql("""
    SELECT * FROM claims LIMIT 10
""").df()

,claim_id,patient_id,provider_id,date_of_service,cpt_code,billed_amount,paid_amount,claim_status,denial_reason
0,CL00001,PA0106,PR011,2026-01-01,71046,337,239,Paid,NaN
1,CL00002,PA0048,PR003,2026-01-01,93306,1015,850,Paid,NaN
2,CL00003,PA0044,PR004,2026-01-01,97140,163,0,Denied,Service not covered
3,CL00004,PA0141,PR010,2026-01-01,99203,204,119,Paid,NaN
4,CL00005,PA0132,PR009,2026-01-01,97110,180,109,Paid,NaN
5,CL00006,PA0108,PR008,2026-01-01,93306,969,580,Paid,NaN
6,CL00007,PA0040,PR005,2026-01-01,85025,59,35,Paid,NaN
7,CL00008,PA0102,PR005,2026-01-02,99214,212,117,Paid,NaN
8,CL00009,PA0177,PR005,2026-01-02,80053,92,66,Paid,NaN
9,CL00010,PA0091,PR009,2026-01-02,97140,156,127,Paid,NaN


### `patients` (200 rows)

In [3]:
duckdb.sql("""
    SELECT * FROM patients LIMIT 10
""").df()

,patient_id,patient_name,date_of_birth,insurance_plan,state
0,PA0001,Laura Lee,1969-01-19,Commercial,TX
1,PA0002,Michael Lopez,1946-07-01,Commercial,AZ
2,PA0003,Susan Johnson,1945-03-15,Commercial,TX
3,PA0004,Jessica Williams,1978-11-26,Medicaid,AZ
4,PA0005,Linda Davis,1958-01-31,Medicare,AZ
5,PA0006,Laura Johnson,1945-07-20,Medicaid,FL
6,PA0007,David Martinez,1944-03-06,Medicaid,NY
7,PA0008,Maria Moore,1952-12-09,Medicaid,FL
8,PA0009,Richard Lopez,1949-03-30,Medicaid,TX
9,PA0010,Patricia Martin,1989-02-20,Commercial,FL


### `providers` (13 rows)

In [4]:
duckdb.sql("""
    SELECT * FROM providers
""").df()

,provider_id,provider_name,specialty,state
0,PR001,Sunrise Family Clinic,Family Medicine,TX
1,PR002,Lakeside Imaging,Radiology,FL
2,PR003,Northside Cardiology,Cardiology,TX
3,PR004,Bayview Physical Therapy,Physical Therapy,CA
4,PR005,Pine Valley Medical Group,Internal Medicine,AZ
5,PR006,Riverside Family Practice,Family Medicine,FL
6,PR007,Desert Sun Radiology,Radiology,AZ
7,PR008,Golden Coast Heart Center,Cardiology,CA
8,PR009,Harbor Rehab & Therapy,Physical Therapy,FL
9,PR010,Maple Street Clinic,Family Medicine,NY


### `procedures` (13 rows)

In [5]:
duckdb.sql("""
    SELECT * FROM procedures
""").df()

,cpt_code,description,service_type,standard_charge
0,99213,"Office visit, established patient, low complexity",Office Visit,150
1,99214,"Office visit, established patient, moderate co...",Office Visit,220
2,99215,"Office visit, established patient, high comple...",Office Visit,300
3,99203,"Office visit, new patient, low complexity",Office Visit,200
4,71046,"Chest X-ray, 2 views",Imaging,320
5,72148,MRI lumbar spine without contrast,Imaging,1200
6,74177,CT abdomen and pelvis with contrast,Imaging,1500
7,93000,"Electrocardiogram (ECG), complete",Cardiology,210
8,93306,"Echocardiogram, complete",Cardiology,950
9,97110,"Therapeutic exercises, 15 minutes",Physical Therapy,180


> 🇪🇸 **Vocabulario nuevo**
>
> | Inglés | Español |
> |---|---|
> | `procedures` | procedimientos (la tabla de códigos CPT) |
> | `description` | descripción del procedimiento |
> | `service_type` | tipo de servicio (*Office Visit* = consulta, *Imaging* = imágenes, *Lab* = laboratorio) |
> | `standard_charge` | precio de lista del código |
> | `date_of_birth` | fecha de nacimiento (en el trabajo le dicen **DOB**) |
> | `state` | estado de EE. UU. (TX = Texas, FL = Florida, CA = California, NY = Nueva York, AZ = Arizona) |
>
> **Ojo, cosas que ya conoces con datos nuevos:** los claims `Pending` tienen `paid_amount` vacío (NULL, se ve como `NaN`), y los `Denied` tienen 0.

---
## Example · SELECT, WHERE, ORDER BY

All claims from provider **PR002** (Lakeside Imaging), oldest first.

> 🇪🇸 **Te recuerdo** la plantilla de siempre: `SELECT` (qué columnas) → `FROM` (qué tabla) → `WHERE` (qué filas) → `ORDER BY` (en qué orden). El texto va con comillas simples.

In [6]:
duckdb.sql("""
    SELECT claim_id, date_of_service, cpt_code, billed_amount, claim_status
    FROM claims
    WHERE provider_id = 'PR002'
    ORDER BY date_of_service
""").df()

,claim_id,date_of_service,cpt_code,billed_amount,claim_status
0,CL00014,2026-01-03,74177,1627,Paid
1,CL00042,2026-01-08,71046,315,Paid
2,CL00048,2026-01-09,72148,1232,Denied
3,CL00094,2026-01-17,72148,1207,Paid
4,CL00123,2026-01-22,72148,1188,Paid
...,...,...,...,...,...
80,CL00914,2026-06-16,74177,1530,Paid
81,CL00951,2026-06-21,71046,310,Denied
82,CL00974,2026-06-25,74177,1637,Paid
83,CL00978,2026-06-26,74177,1564,Denied


## Exercise 1 · Denied claims

List all **denied** claims, oldest first.

Columns: `claim_id`, `date_of_service`, `cpt_code`, `billed_amount`, `denial_reason`.

✔ **174 rows.** The first one is **CL00003** (2026-01-01, CPT 97140, $163, *Service not covered*).

> 🇪🇸 Muestra todos los claims **rechazados** (`claim_status` = `'Denied'`), del más antiguo al más nuevo, con esas 5 columnas.

In [16]:
duckdb.sql("""
    SELECT claim_id, date_of_service, cpt_code, billed_amount, denial_reason
    FROM claims
    WHERE claim_status = 'Denied'
    ORDER BY date_of_service
     
""").df()

,claim_id,date_of_service,cpt_code,billed_amount,denial_reason
0,CL00003,2026-01-01,97140,163,Service not covered
1,CL00017,2026-01-03,97110,191,Missing prior authorization
2,CL00023,2026-01-05,74177,1543,Missing prior authorization
3,CL00027,2026-01-05,74177,1632,Missing prior authorization
4,CL00029,2026-01-06,74177,1475,Service not covered
...,...,...,...,...,...
169,CL00983,2026-06-27,97140,162,Service not covered
170,CL00984,2026-06-28,99214,210,Patient not eligible
171,CL00987,2026-06-28,97110,185,Service not covered
172,CL00988,2026-06-28,99213,148,Patient not eligible


---
## New: LIMIT

`LIMIT` shows only the **first N rows** of the result.

> 🇪🇸 **Lo nuevo:** `LIMIT 5` = "muéstrame solo las primeras 5 filas". Va **al final**, después del `ORDER BY`.
> Juntos sirven para los "top": primero ordenas y después cortas. En Excel es como ordenar una columna y mirar solo las primeras filas.
>
> Para ordenar de **mayor a menor** se agrega `DESC` (*descending*, descendente). Sin `DESC`, el orden es de menor a mayor.

**Example:** the 3 cheapest claims.

In [8]:
duckdb.sql("""
    SELECT claim_id, cpt_code, billed_amount
    FROM claims
    ORDER BY billed_amount
    LIMIT 3
""").df()

,claim_id,cpt_code,billed_amount
0,CL00015,85025,57
1,CL00148,85025,57
2,CL00246,85025,57


## Exercise 2 · Top 5 most expensive claims

Show the **5 claims with the highest** `billed_amount`, from highest to lowest.

Columns: `claim_id`, `cpt_code`, `billed_amount`, `claim_status`.

✔ **5 rows.** The first one is **CL00756** ($1,650). All 5 are CPT **74177** (CT scan): **1 is Denied** (CL00210) and **1 is Pending** (CL00763).

> 🇪🇸 Los 5 claims **más caros** (mayor `billed_amount`). Pista: es como el ejemplo, pero de mayor a menor (`DESC`).

In [25]:
duckdb.sql("""
    SELECT claim_id, cpt_code, billed_amount, claim_status
    FROM claims
    ORDER BY billed_amount DESC
    LIMIT 5
""").df()

,claim_id,cpt_code,billed_amount,claim_status
0,CL00756,74177,1650,Paid
1,CL00451,74177,1648,Paid
2,CL00880,74177,1648,Paid
3,CL00210,74177,1646,Denied
4,CL00763,74177,1642,Pending


## Exercise 3 · Imaging procedures

From the `procedures` table, list the **Imaging** procedures, from the highest to the lowest `standard_charge`.

Columns: `cpt_code`, `description`, `standard_charge`.

✔ **3 rows.** The first one is **74177** (*CT abdomen and pelvis with contrast*, $1,500).

> 🇪🇸 Ojo: esta vez la tabla es **`procedures`**, no `claims`. Filtra por `service_type` = `'Imaging'`.

In [19]:
duckdb.sql("""
    SELECT cpt_code, description, standard_charge
    FROM procedures
    WHERE service_type = 'Imaging' 
    ORDER BY standard_charge DESC
""").df()

,cpt_code,description,standard_charge
0,74177,CT abdomen and pelvis with contrast,1500
1,72148,MRI lumbar spine without contrast,1200
2,71046,"Chest X-ray, 2 views",320


## Exercise 4 · High-value denials

List the **denied** claims with a `billed_amount` **greater than 1000**, from the highest to the lowest amount.

Columns: `claim_id`, `date_of_service`, `cpt_code`, `billed_amount`, `denial_reason`.

✔ **54 rows.** The first one is **CL00210** ($1,646, *Missing prior authorization*).

> 🇪🇸 Dos condiciones en el `WHERE`, unidas con `AND`: rechazado **y** monto mayor a 1000.
> Cuando lo tengas, mira la columna `denial_reason`: ¿qué motivo se repite más? Eso es un hallazgo de negocio.

In [21]:
duckdb.sql("""
    SELECT claim_id, date_of_service, cpt_code, billed_amount, denial_reason
    FROM claims
    WHERE claim_status = 'Denied' AND billed_amount > 1000
    ORDER BY billed_amount DESC
""").df()

,claim_id,date_of_service,cpt_code,billed_amount,denial_reason
0,CL00210,2026-02-06,74177,1646,Missing prior authorization
1,CL00027,2026-01-05,74177,1632,Missing prior authorization
2,CL00192,2026-02-01,74177,1630,Missing prior authorization
3,CL00579,2026-04-10,74177,1622,Service not covered
4,CL00342,2026-02-28,74177,1620,Missing prior authorization
5,CL00097,2026-01-18,74177,1616,Missing prior authorization
6,CL00335,2026-02-26,74177,1603,Missing prior authorization
7,CL00362,2026-03-04,74177,1595,Service not covered
8,CL00203,2026-02-03,74177,1565,Service not covered
9,CL00978,2026-06-26,74177,1564,Missing prior authorization


## Exercise 5 · Medicaid patients in Texas

List the patients with **Medicaid** who live in **TX**, sorted by `patient_name` (A to Z).

Columns: `patient_id`, `patient_name`, `date_of_birth`.

✔ **9 rows.** The first one is **Ashley Harris** (PA0131).

> 🇪🇸 Tabla `patients`. Dos condiciones con `AND`: `insurance_plan` y `state`.
> Cuando lo tengas, mira bien los nombres: hay dos **Ashley Williams**. ¿Son la misma persona? ¿Cómo lo sabes?

In [26]:
duckdb.sql("""
    SELECT patient_id, patient_name, date_of_birth
    FROM patients
    WHERE insurance_plan = 'Medicaid' AND state = 'TX' 
    ORDER BY patient_name
""").df()

,patient_id,patient_name,date_of_birth
0,PA0131,Ashley Harris,1965-08-13
1,PA0019,Ashley Williams,1993-04-28
2,PA0194,Ashley Williams,1943-08-25
3,PA0066,Diego Taylor,1951-09-24
4,PA0119,Joseph Williams,2001-08-02
5,PA0188,Michael Smith,1969-10-04
6,PA0009,Richard Lopez,1949-03-30
7,PA0036,Sofia Martin,1953-07-27
8,PA0147,Susan White,1996-08-07


---
## Check your understanding

> 🇪🇸 Responde en el chat, con tus palabras:
> 1. ¿Qué pasa si pones `LIMIT 10` **sin** `ORDER BY`? ¿Te da los 5 más caros?
> 2. ¿Por qué en un sistema de claims se usa `patient_id` y no el nombre para identificar a un paciente?